# exp_7 · 02 evaluation — nested decoder selection (IDSSE only, never fits)
1. **Recheck (binding).** exp_7's `evaluate` on exp_2's cached `exp3_ref_ens3` `target_like` predictions reproduces
   exp_3's stored rows, selections and final configs.
2. Every pre-registered (variant, selection view, label set): selection view = deployment view; the held-out match is
   scored on every view decoded with that selection. Results: `results/variant_*__sel_*__*.json`.

In [1]:
from pathlib import Path
import os, sys, json, time, subprocess
from datetime import datetime, timezone
ROOT = Path.cwd()
if not (ROOT / 'exp7_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_7')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import exp7_common as C
import exp1_common as C1
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
def now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')
def make_log(name):
    path = C.LOG_DIR / f'{name}_progress.log'
    path.parent.mkdir(parents=True, exist_ok=True)
    def log(message, **_):
        print(message, flush=True)
        with path.open('a') as handle:
            handle.write(f'{now()} {message}\n')
    return log
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])

GUARD_FILE = 'guard_02.json'
guard = C.guard_check(C.RESULTS / 'write_guard_before.json', C.RESULTS / GUARD_FILE)
print('nothing written under exp_0-exp_6 / data since freezing:', guard['all_unchanged'],
      {k: v['unchanged'] for k, v in guard['folders'].items()})
if not guard['all_unchanged']:
    raise RuntimeError('Write guard changed: diagnose before continuing')

import exp7_eval as E
log = make_log('02')

PREREG verified 189874eb7d34ab4c | frozen 2026-11-11T01:29:38+00:00


nothing written under exp_0-exp_6 / data since freezing: True {'exp_0': True, 'exp_1': True, 'exp_2': True, 'exp_3': True, 'exp_4': True, 'exp_5': True, 'exp_6': True, 'data': True, 'data_metric_repo': True}


In [2]:
rechecks = [E.evaluation_path_recheck(label) for label in C.LABEL_SETS]
C1.write_json(C.RESULTS / 'eval_recheck.json', {'checked_at_utc': now(), 'rechecks': rechecks})
print(rechecks)
if not all(r['rows_equal'] and r['selections_equal'] and r['final_config_equal'] for r in rechecks):
    raise RuntimeError('Binding recheck failed: exp_7 evaluation path does not reproduce exp_3')

[{'label_set': 'provider', 'rows_equal': True, 'selections_equal': True, 'final_config_equal': True}, {'label_set': 'ctrl', 'rows_equal': True, 'selections_equal': True, 'final_config_equal': True}]


In [3]:
cache, timings = {}, []
for task in E.tasks():
    seconds, tag = E.evaluate_and_save(task, cache)
    timings.append({'task': list(task), 'seconds': round(seconds, 1), 'tag': tag})
    rows = C1.read_json(E.result_path(*task))['rows']
    means = pd.DataFrame(rows).groupby('view')['spell_f1'].mean().round(4).to_dict()
    log(f'EVAL {task} {tag} {seconds:.0f}s {means}')
C1.write_json(C.RESULTS / 'evaluation.json', {'finished_at_utc': now(), 'tasks': timings})

EVAL ('exp7_robust_ens3', 'target_like', 'provider') computed 80s {'target_like': 0.6634}


EVAL ('exp7_robust_ens3', 'target_like', 'ctrl') computed 80s {'target_like': 0.7424}


EVAL ('exp7_robust_ens3', 'measured', 'provider') computed 95s {'ar1': 0.6781, 'measured': 0.6779, 'stress': 0.66}


EVAL ('exp7_robust_ens3', 'measured', 'ctrl') computed 82s {'ar1': 0.7515, 'measured': 0.7505, 'stress': 0.7429}


EVAL ('exp7_robust_ens3', 'clean25', 'provider') computed 95s {'clean25': 0.6879}


EVAL ('exp7_robust_ens3', 'clean25', 'ctrl') computed 81s {'clean25': 0.7652}


... (12 outputs left out)

EVAL ('exp7_robust_s2', 'measured', 'provider') computed 95s {'ar1': 0.6633, 'measured': 0.6635, 'stress': 0.6426}


EVAL ('exp7_robust_s2', 'measured', 'ctrl') computed 82s {'ar1': 0.7436, 'measured': 0.7424, 'stress': 0.7266}


EVAL ('exp3_ref_ens3_msel', 'measured', 'provider') computed 300s {'ar1': 0.5964, 'measured': 0.574, 'stress': 0.5761}


EVAL ('exp3_ref_ens3_msel', 'measured', 'ctrl') computed 105s {'ar1': 0.6643, 'measured': 0.6381, 'stress': 0.6568}
